# Failure B Resolution Suite

Notebook nay chi tap trung vao `Failure B`:

- `ultra-center over-confidence`
- tach rieng:
  - center amplitude / dispersion
  - center direction / wrong-sign

Muc tieu cua suite:

1. `D1`: audit do sach cua `raw train-center labels` so voi `oracle-center`
2. `D2`: audit gradient interference de xem update tu non-center bands co lam center logits no len hay khong
3. `R1`: replicate control cho `baseline / L0 / L1`
4. chay 2 pilot sua Failure B:
   - `P_B2_raw_center_strong`
   - `P_B1_proxy_center_weighted`

Nguyen tac:

- khong dung `MSE` tong lam metric chon winner
- toan bo output deu ghi ra file rieng va co the tai su dung neu notebook bi crash
- notebook nay chay tren repo day du o `C:\Users\USER\Desktop\chess_engine`

In [1]:
from dataclasses import replace
from pathlib import Path
import sys

import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path(r"C:\Users\USER\Desktop\chess_engine")
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / "failure_b_resolution_suite"
DATA_ROOT = PROJECT_ROOT / "data" / "process"
RUN_DIR = Path(r"C:\Users\USER\Downloads\dgrn_5m_v3_stage2_polish_run1")

if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

import failure_b_resolution_helpers as lab

torch.set_float32_matmul_precision("high")
lab.set_global_seed(123)
DEVICE = lab.choose_device(prefer_cuda=True)
if DEVICE.type != "cuda":
    raise RuntimeError("CUDA is required.")
if "envs\\chess_engine" not in sys.executable.lower():
    raise RuntimeError(f"Wrong interpreter: {sys.executable}")

paths = lab.build_default_paths(run_dir=RUN_DIR, data_root=DATA_ROOT, experiment_dir=EXPERIMENT_DIR)
lab.export_paths_json(paths, paths["output_dir"] / "paths.json")
runtime_check = lab.validate_runtime_paths(paths)
CHECKPOINT = paths["run_dir"] / "ckpt_best.pt"
L3_CHECKPOINT = paths["objective_output_dir"] / "runs" / "L3_full_A1" / "checkpoints" / "L3_full_A1_best.pt"

CENTER_CFG = lab.CenterPurityConfig()
GRAD_CFG = lab.GradientAuditConfig(
    train_split="train",
    train_num_shards=8,
    batch_size=256,
    batches_per_band=3,
    sample_seed=123,
    influence_step_l2=5e-4,
    probe_center_max=128,
    probe_midband_max=128,
)
CONTROL_REPL_CFG = lab.obj_lab.ReplicateOracleConfig(
    split="test",
    sample_abs_y_edges=(0.0, 0.05, 0.20, 0.50, 0.70, 1.00),
    sample_per_band=24,
    num_replicates=3,
    base_seed=1777,
)
PILOT_CFG = lab.PilotTrainConfig(
    batch_size=576,
    epochs=1,
    learning_rate=3e-6,
    min_lr=1e-6,
    weight_decay=2e-4,
    grad_clip_norm=1.0,
    seed=123,
    train_num_shards=8,
    test_max_samples=200_000,
    test_num_shards=4,
    log_every_steps=200,
    max_mem_ratio=0.85,
    min_batch_size=128,
    batch_step=64,
)
ORACLE_CFG = lab.ab_lab.OracleEvalConfig()
EVAL_CFG = lab.ab_lab.TrainConfig(
    batch_size=PILOT_CFG.batch_size,
    epochs=1,
    learning_rate=3e-6,
    min_lr=1e-6,
    weight_decay=2e-4,
    grad_clip_norm=1.0,
    seed=123,
    log_every_steps=200,
    train_num_shards=None,
    val_max_samples=100_000,
    test_max_samples=200_000,
    val_num_shards=2,
    test_num_shards=4,
    benchmark_num_shards=1,
)

AUTOTUNE = lab.obj_lab.autotune_train_batch_size(
    init_ckpt_path=L3_CHECKPOINT,
    data_root=DATA_ROOT,
    device=DEVICE,
    preferred_batch_size=PILOT_CFG.batch_size,
    min_batch_size=PILOT_CFG.min_batch_size,
    step=PILOT_CFG.batch_step,
    max_mem_ratio=PILOT_CFG.max_mem_ratio,
)
PILOT_CFG = replace(PILOT_CFG, batch_size=int(AUTOTUNE["selected_batch_size"]))
EVAL_CFG = replace(EVAL_CFG, batch_size=int(AUTOTUNE["selected_batch_size"]))
lab.save_json(AUTOTUNE, paths["reports_dir"] / "pilot_batch_autotune.json")

ORACLE_BUNDLE = lab.ab_lab.load_oracle_subset_bundle(ORACLE_CFG, data_root=DATA_ROOT)
EXISTING_REGISTRY_DF = lab.load_checkpoint_registry(paths)
EXISTING_REGISTRY = EXISTING_REGISTRY_DF.to_dict("records")
PILOT_VARIANTS = lab.build_pilot_variants(paths)

display(pd.DataFrame({"path_key": list(paths.keys()), "path_value": [str(v) for v in paths.values()]}))
display(pd.DataFrame([runtime_check]))
display(pd.DataFrame([AUTOTUNE]))
display(EXISTING_REGISTRY_DF)

,path_key,path_value
0,project_root,C:\Users\USER\Desktop\chess_engine
1,experiment_dir,C:\Users\USER\Desktop\chess_engine\experiments...
2,output_dir,C:\Users\USER\Desktop\chess_engine\experiments...
3,reports_dir,C:\Users\USER\Desktop\chess_engine\experiments...
4,plots_dir,C:\Users\USER\Desktop\chess_engine\experiments...
5,cache_dir,C:\Users\USER\Desktop\chess_engine\experiments...
6,pilots_dir,C:\Users\USER\Desktop\chess_engine\experiments...
7,replicates_dir,C:\Users\USER\Desktop\chess_engine\experiments...
8,data_root,C:\Users\USER\Desktop\chess_engine\data\process
9,run_dir,C:\Users\USER\Downloads\dgrn_5m_v3_stage2_poli...


,is_valid,stockfish_path,baseline_ckpt,l3_ckpt
0,True,D:\stockfish-windows-x86-64-avx2\stockfish\sto...,C:\Users\USER\Downloads\dgrn_5m_v3_stage2_poli...,C:\Users\USER\Desktop\chess_engine\experiments...


,selected_batch_size,preferred_batch_size,min_batch_size,step,max_mem_ratio,device,total_mem_gb,selected_report,attempts
0,576,576,128,64,0.85,cuda,3.999512,"{'batch_size': 576, 'step_time_sec': 3.9920680...","[{'batch_size': 576, 'ok': True, 'peak_mem_gb'..."


,label,checkpoint,target_scale,source
0,baseline,C:\Users\USER\Downloads\dgrn_5m_v3_stage2_poli...,600.0,stage2_polish
1,A2_band_balanced,C:\Users\USER\Desktop\chess_engine\experiments...,600.0,root_cause_ablation_suite
2,L0_control_hybrid,C:\Users\USER\Desktop\chess_engine\experiments...,600.0,objective_resolution_suite
3,L1_z_strong_hybrid,C:\Users\USER\Desktop\chess_engine\experiments...,600.0,objective_resolution_suite
4,L3_full_A1,C:\Users\USER\Desktop\chess_engine\experiments...,600.0,objective_resolution_suite
5,L4_A1_plus_A2,C:\Users\USER\Desktop\chess_engine\experiments...,600.0,objective_resolution_suite
6,S1_A1_center_w020_m010,C:\Users\USER\Desktop\chess_engine\experiments...,600.0,objective_resolution_suite


## D1: Center Label Purity Audit

Cell nay gop tat ca oracle subset da co, deduplicate theo `(split, shard_id, local_index)`,
roi tra loi 2 cau hoi:

1. `|y_train| <= tau` co thuc su map vao `oracle-center stable` hay khong?
2. co the xay `trusted-center proxy` tu `(abs_y, abs_pred, abs_err)` hay khong?

In [2]:
center_pool = lab.build_oracle_center_pool(paths, CENTER_CFG, refresh=False)
purity_audit = lab.run_center_label_purity_audit(center_pool["unique_rows"], CENTER_CFG, paths)
center_lookup = lab.build_center_purity_lookup(center_pool["unique_rows"], CENTER_CFG, paths)

display(pd.DataFrame([center_pool["report"]]))
display(purity_audit["summary"])
display(purity_audit["band_summary"])
display(center_lookup["lookup"].sort_values("smoothed_clean_rate", ascending=False).head(20))

,num_sources,num_rows_all,num_rows_unique,num_conflict_rows,conflict_rows_preview,source_counts,clean_center_rate_unique
0,5,720,480,0,[],"{'oracle_root_cause_primary': 240, 'objective_...",0.045833


,raw_center_thr,oracle_center_thr,n,raw_center_count,oracle_center_clean_count,tp,fp,fn,tn,precision,recall,specificity,oracle_clean_rate_inside_raw_center,mean_abs_oracle_inside_raw_center,stable_rate_inside_raw_center
0,0.02,0.05,480,60,22,16,44,6,414,0.266667,0.727273,0.903930,0.266667,0.029591,0.366667
1,0.02,0.10,480,60,38,21,39,17,403,0.350000,0.552632,0.911765,0.350000,0.029591,0.366667
2,0.05,0.05,480,96,22,21,75,1,383,0.218750,0.954545,0.836245,0.218750,0.039687,0.333333
3,0.05,0.10,480,96,38,30,66,8,376,0.312500,0.789474,0.850679,0.312500,0.039687,0.333333
4,0.10,0.05,480,147,22,22,125,0,333,0.149660,1.000000,0.727074,0.149660,0.060065,0.319728
5,0.10,0.10,480,147,38,38,109,0,333,0.258503,1.000000,0.753394,0.258503,0.060065,0.319728


,raw_band,n,mean_abs_oracle,stable_rate,oracle_clean_rate_005,oracle_clean_rate_010
0,"[0.000,0.020]",60,0.029591,0.366667,0.266667,0.350000
1,"[0.020,0.050]",36,0.056513,0.277778,0.138889,0.250000
2,"[0.050,0.100]",51,0.098425,0.294118,0.019608,0.156863
3,"[0.100,0.200]",45,0.154347,0.377778,0.000000,0.000000
4,"[0.200,1.010]",288,0.511071,0.333333,0.000000,0.000000


,bin_abs_y,bin_abs_pred,bin_abs_err,count,clean_count,clean_rate,smoothed_clean_rate
0,0,0,0,19,6,0.315789,0.266566
1,0,0,1,1,1,1.000000,0.244142
2,0,1,1,10,3,0.300000,0.233182
3,0,2,1,10,3,0.300000,0.233182
4,0,2,2,8,2,0.250000,0.199830
7,1,0,1,3,1,0.333333,0.199753
11,1,2,2,4,1,0.250000,0.183107
12,1,3,2,10,2,0.200000,0.177627
17,2,1,2,5,1,0.200000,0.169021
18,2,2,0,1,0,0.000000,0.133031


## D2: Gradient Interference Audit

Cell nay khong train.

No do 3 lop thong tin:

- gradient norm theo band
- cosine similarity giua gradient center va gradient mid-band
- one-step influence:
  mot buoc update tu batch nao lam `|pred|` tren `oracle-center clean probe` tang bao nhieu

In [3]:
probe_sets = lab.build_probe_sets(center_pool["unique_rows"], DATA_ROOT, GRAD_CFG, paths, refresh=False)
grad_manifest = lab.sample_gradient_batches(DATA_ROOT, GRAD_CFG, paths, refresh=False)
gradient_batches = lab.load_gradient_batches(DATA_ROOT, grad_manifest, paths)
grad_audit = lab.run_gradient_interference_audit(
    checkpoint_path=CHECKPOINT,
    gradient_batches=gradient_batches,
    probe_sets=probe_sets,
    cfg=GRAD_CFG,
    paths=paths,
    device=DEVICE,
    refresh=False,
)

display(pd.DataFrame([grad_audit["summary"]]))
display(grad_audit["norms"])
display(grad_audit["cosines"])
display(grad_audit["influence"].sort_values(["probe_name", "delta_mean_abs_pred"], ascending=[True, False]))

,baseline_checkpoint,objectives,bands,probe_names,influence_step_l2,center_probe_delta_mean_abs_pred_from_midbands
0,C:\Users\USER\Downloads\dgrn_5m_v3_stage2_poli...,"[baseline_obj, a1_obj]","[center_raw_0_005, near_center_005_02, mid_02_...","[center_clean_005, center_raw_mismatch, midban...",0.0005,"{'baseline_obj': 0.0023285703947240055, 'a1_ob..."


,grad_path,band_name,objective_name,grad_norm_stem,grad_norm_backbone,grad_norm_head,grad_norm_all
0,C:\Users\USER\Desktop\chess_engine\experiments...,center_raw_0_005,baseline_obj,0.981554,0.110869,0.625076,1.168957
1,C:\Users\USER\Desktop\chess_engine\experiments...,near_center_005_02,baseline_obj,0.549295,0.063178,0.359295,0.659401
2,C:\Users\USER\Desktop\chess_engine\experiments...,mid_02_05,baseline_obj,0.523803,0.075813,0.436249,0.685879
3,C:\Users\USER\Desktop\chess_engine\experiments...,mid_05_07,baseline_obj,1.021867,0.133086,0.758290,1.279424
4,C:\Users\USER\Desktop\chess_engine\experiments...,center_raw_0_005,a1_obj,0.989883,0.098825,0.548190,1.135846
5,C:\Users\USER\Desktop\chess_engine\experiments...,near_center_005_02,a1_obj,0.581089,0.056218,0.304951,0.658650
6,C:\Users\USER\Desktop\chess_engine\experiments...,mid_02_05,a1_obj,0.612218,0.069039,0.371843,0.719614
7,C:\Users\USER\Desktop\chess_engine\experiments...,mid_05_07,a1_obj,1.006668,0.121443,0.698269,1.231141


,objective_name,band_left,band_right,cosine_all,cosine_backbone,cosine_head
0,baseline_obj,center_raw_0_005,center_raw_0_005,1.003180,1.000930,1.000021
1,baseline_obj,center_raw_0_005,near_center_005_02,0.677974,0.301437,0.389371
2,baseline_obj,center_raw_0_005,mid_02_05,-0.239809,-0.091335,-0.025953
3,baseline_obj,center_raw_0_005,mid_05_07,-0.642308,-0.279241,-0.723877
4,baseline_obj,near_center_005_02,center_raw_0_005,0.677974,0.301437,0.389371
5,baseline_obj,near_center_005_02,near_center_005_02,1.003857,1.000761,1.000020
6,baseline_obj,near_center_005_02,mid_02_05,-0.455284,-0.096561,-0.746055
7,baseline_obj,near_center_005_02,mid_05_07,-0.554917,-0.253906,-0.649672
8,baseline_obj,mid_02_05,center_raw_0_005,-0.239809,-0.091335,-0.025953
9,baseline_obj,mid_02_05,near_center_005_02,-0.455284,-0.096561,-0.746055


,objective_name,source_band,probe_name,step_l2,before_mean_abs_pred,after_mean_abs_pred,delta_mean_abs_pred,before_mae_vs_oracle,after_mae_vs_oracle,delta_mae_vs_oracle,before_center_false_0.1,after_center_false_0.1,delta_center_false_0.1
18,a1_obj,mid_02_05,center_clean_005,0.0005,0.098740,0.102256,0.003515,0.094196,0.098677,0.004481,0.409091,0.409091,0.000000
21,a1_obj,mid_05_07,center_clean_005,0.0005,0.098740,0.101513,0.002773,0.094196,0.097872,0.003676,0.409091,0.409091,0.000000
9,baseline_obj,mid_05_07,center_clean_005,0.0005,0.098740,0.101265,0.002525,0.094196,0.097480,0.003283,0.409091,0.409091,0.000000
6,baseline_obj,mid_02_05,center_clean_005,0.0005,0.098740,0.100873,0.002132,0.094196,0.097004,0.002808,0.409091,0.454545,0.045455
3,baseline_obj,near_center_005_02,center_clean_005,0.0005,0.098740,0.098421,-0.000319,0.094196,0.093454,-0.000742,0.409091,0.409091,0.000000
15,a1_obj,near_center_005_02,center_clean_005,0.0005,0.098740,0.098403,-0.000337,0.094196,0.093949,-0.000247,0.409091,0.409091,0.000000
12,a1_obj,center_raw_0_005,center_clean_005,0.0005,0.098740,0.098136,-0.000605,0.094196,0.093470,-0.000726,0.409091,0.409091,0.000000
0,baseline_obj,center_raw_0_005,center_clean_005,0.0005,0.098740,0.097993,-0.000747,0.094196,0.093351,-0.000845,0.409091,0.409091,0.000000
22,a1_obj,mid_05_07,center_raw_mismatch,0.0005,0.062570,0.064914,0.002344,0.099638,0.099147,-0.000491,NaN,NaN,NaN
10,baseline_obj,mid_05_07,center_raw_mismatch,0.0005,0.062570,0.064788,0.002218,0.099638,0.099180,-0.000458,NaN,NaN,NaN


## R1: Replicate Controls For Baseline / L0 / L1

Cell nay chi tra loi attribution:

- train them 1 epoch voi objective cu (`L0`) co tu no cai thien khong?
- neu co, `L1` co them contribution rieng tu z-space hay khong?

In [4]:
control_repl = lab.run_control_replicate_l0_l1(
    baseline_ckpt=CHECKPOINT,
    data_root=DATA_ROOT,
    cfg=CONTROL_REPL_CFG,
    paths=paths,
    device=DEVICE,
    refresh=False,
)
display(control_repl["aggregate"].sort_values("oracle_teacher_mae_mean", ascending=True))

[test_pred_cache_00000] offset=0 / 50000 elapsed=1.1s
[test-pred-cache] shards=1/10 elapsed=25.9s
[test_pred_cache_00001] offset=0 / 50000 elapsed=1.1s
[test_pred_cache_00002] offset=0 / 50000 elapsed=1.1s
[test_pred_cache_00003] offset=0 / 50000 elapsed=1.1s
[test_pred_cache_00004] offset=0 / 50000 elapsed=1.1s
[test-pred-cache] shards=5/10 elapsed=129.2s
[test_pred_cache_00005] offset=0 / 50000 elapsed=1.1s
[test_pred_cache_00006] offset=0 / 50000 elapsed=1.1s
[test_pred_cache_00007] offset=0 / 50000 elapsed=1.1s
[test_pred_cache_00008] offset=0 / 50000 elapsed=1.1s
[test-pred-cache] shards=9/10 elapsed=232.7s
[test_pred_cache_00009] offset=0 / 50000 elapsed=1.1s
[oracle-subset] processed=1/120
[oracle-subset] processed=13/120
[oracle-subset] processed=25/120
[oracle-subset] processed=37/120
[oracle-subset] processed=49/120
[oracle-subset] processed=61/120
[oracle-subset] processed=73/120
[oracle-subset] processed=85/120
[oracle-subset] processed=97/120
[oracle-subset] processed=109/

,label,oracle_teacher_mae_mean,oracle_teacher_mae_std,oracle_closer_rate_mean,oracle_closer_rate_std,oracle_stable_0.7_slope_mean,oracle_stable_0.7_slope_std,oracle_midband_mae_sum_stable_mean,oracle_midband_mae_sum_stable_std,oracle_center_amp_ratio_mean,...,oracle_center_wrong_sign_0.1eq_mean,oracle_center_wrong_sign_0.1eq_std,oracle_center_wrong_sign_0.2eq_mean,oracle_center_wrong_sign_0.2eq_std,oracle_center_spread_ratio_mean,oracle_center_spread_ratio_std,oracle_band_amp_0_0.05_stable_mean,oracle_band_amp_0_0.05_stable_std,oracle_band_sign_0.05_0.2_stable_mean,oracle_band_sign_0.05_0.2_stable_std
0,L0_control_hybrid,0.169495,0.004799,0.238889,0.031549,0.742813,0.137321,0.475257,0.087328,5.421480,...,0.090498,0.023512,0.0,0.0,6.991404,3.350301,7.286520,5.763486,0.844444,0.150308
1,L1_z_strong_hybrid,0.170390,0.002808,0.236111,0.026788,0.754754,0.137271,0.476145,0.088000,5.624625,...,0.090498,0.023512,0.0,0.0,7.340185,3.486125,7.962152,6.731065,0.844444,0.150308
2,baseline,0.171437,0.006184,0.247222,0.033679,0.741510,0.134521,0.478982,0.086467,5.534212,...,0.084465,0.088477,0.0,0.0,7.314330,3.636215,7.169664,6.019865,0.844444,0.150308


## Evaluate Existing Runs On Pooled Center Bundle

Cell nay tao mot pooled stable-center bundle tu tat ca oracle subsets da co,
roi chấm lai cac checkpoint hien co theo metric Failure B.

In [5]:
pooled_center_bundle = lab.build_pooled_center_bundle(
    pooled_unique=center_pool["unique_rows"],
    data_root=DATA_ROOT,
    center_thr=CENTER_CFG.oracle_center_thr,
    refresh=False,
    paths=paths,
)
existing_eval = lab.evaluate_failure_b_registry(
    registry=EXISTING_REGISTRY,
    pooled_center_bundle=pooled_center_bundle,
    oracle_bundle=ORACLE_BUNDLE,
    data_root=DATA_ROOT,
    train_cfg=EVAL_CFG,
    oracle_cfg=ORACLE_CFG,
    paths=paths,
    device=DEVICE,
    prefix="existing_failure_b",
)
display(existing_eval["primary"])

[test_600] offset=0 / 200000 elapsed=0.6s
[test_600] offset=25600 / 200000 elapsed=13.9s
[test_600] offset=51200 / 200000 elapsed=27.1s
[test_600] offset=76800 / 200000 elapsed=40.3s
[test_600] offset=102400 / 200000 elapsed=53.6s
[test_600] offset=128000 / 200000 elapsed=66.8s
[test_600] offset=153600 / 200000 elapsed=80.1s
[test_600] offset=179200 / 200000 elapsed=93.4s
[oracle_subset_600] offset=0 / 240 elapsed=0.1s
[test_600] offset=0 / 200000 elapsed=0.5s
[test_600] offset=25600 / 200000 elapsed=13.8s
[test_600] offset=51200 / 200000 elapsed=27.1s
[test_600] offset=76800 / 200000 elapsed=40.4s
[test_600] offset=102400 / 200000 elapsed=53.6s
[test_600] offset=128000 / 200000 elapsed=66.9s
[test_600] offset=153600 / 200000 elapsed=80.2s
[test_600] offset=179200 / 200000 elapsed=93.5s
[oracle_subset_600] offset=0 / 240 elapsed=0.1s
[test_600] offset=0 / 200000 elapsed=0.5s
[test_600] offset=25600 / 200000 elapsed=13.8s
[test_600] offset=51200 / 200000 elapsed=27.1s
[test_600] offset=

,label,target_scale,metric_scale,test_mse_0.1eq,test_mse_0.2eq,test_mse_0.5eq,test_mse_0.7eq,test_slope_0.1eq,test_slope_0.2eq,test_slope_0.7eq,...,oracle_band_sign_0.2_0.5_stable,selection_score_v2,pooled_center_mae,pooled_center_amp_ratio,pooled_center_false_0.1eq,pooled_center_false_0.2eq,pooled_center_wrong_sign_0.1eq,pooled_center_wrong_sign_0.2eq,pooled_center_spread_ratio,failure_b_score
0,A2_band_balanced,600.0,600.0,0.029684,0.028911,0.036876,0.050254,1.283758,0.898517,0.594372,...,1.000000,1.462451,0.092978,4.145144,0.363636,0.136364,0.045455,0.0,4.354346,0.617330
1,L0_control_hybrid,600.0,600.0,0.031418,0.030474,0.038108,0.051023,1.336287,0.927135,0.608344,...,1.000000,1.471977,0.092319,4.189385,0.363636,0.136364,0.045455,0.0,4.406538,0.618233
2,baseline,600.0,600.0,0.032140,0.031037,0.038544,0.051428,1.338430,0.923810,0.605962,...,0.933333,1.502943,0.094196,4.317992,0.409091,0.136364,0.045455,0.0,4.468379,0.650608
3,L1_z_strong_hybrid,600.0,600.0,0.034724,0.033309,0.040471,0.052911,1.366364,0.941777,0.617748,...,1.000000,1.478411,0.094821,4.255428,0.454545,0.136364,0.045455,0.0,4.459377,0.653007
4,L4_A1_plus_A2,600.0,600.0,0.039988,0.038093,0.044461,0.055779,1.436736,0.987802,0.645502,...,1.000000,1.551388,0.108972,4.799236,0.500000,0.181818,0.090909,0.0,5.045348,0.732969
5,S1_A1_center_w020_m010,600.0,600.0,0.041476,0.039600,0.045768,0.056668,1.490324,1.023160,0.661409,...,1.000000,1.577594,0.109851,4.897958,0.500000,0.181818,0.090909,0.0,5.114964,0.742564
6,L3_full_A1,600.0,600.0,0.047132,0.044530,0.049749,0.059749,1.575222,1.070809,0.683185,...,1.000000,1.633412,0.121076,5.379499,0.500000,0.227273,0.090909,0.0,5.547908,0.804864


## Pilot Fixes For Failure B

Hai pilot nay test hai huong khac nhau:

- `P_B2_raw_center_strong`: neu Failure B chu yeu do objective pressure / gradient interference
- `P_B1_proxy_center_weighted`: neu raw center labels bi ban, chi nen regularize tren trusted-center proxy

Cell nay resume-safe:

- train prediction cache ghi theo shard
- pilot nao da co `best checkpoint` hop le se duoc skip

In [6]:
pred_cache_manifest = lab.precompute_train_prediction_cache(
    checkpoint_path=L3_CHECKPOINT,
    data_root=DATA_ROOT,
    split="train",
    num_shards=PILOT_CFG.train_num_shards,
    paths=paths,
    device=DEVICE,
    batch_size=max(PILOT_CFG.batch_size, 1024),
    refresh=False,
)

pilot_results = {}
for variant_name in ["P_B2_raw_center_strong", "P_B1_proxy_center_weighted"]:
    pilot_results[variant_name] = lab.run_failure_b_pilot(
        variant=PILOT_VARIANTS[variant_name],
        pilot_cfg=PILOT_CFG,
        data_root=DATA_ROOT,
        oracle_cfg=ORACLE_CFG,
        oracle_bundle=ORACLE_BUNDLE,
        pooled_center_bundle=pooled_center_bundle,
        center_lookup=center_lookup,
        pred_cache_manifest=pred_cache_manifest,
        paths=paths,
        device=DEVICE,
        refresh=False,
    )
pilot_history = pd.concat(
    [result["history"].assign(variant=name) for name, result in pilot_results.items()],
    ignore_index=True,
)
display(pilot_history)

[predict] offset=0 / 50000 elapsed=0.5s
[predict] offset=25600 / 50000 elapsed=13.9s
[predict] offset=0 / 50000 elapsed=0.5s
[predict] offset=25600 / 50000 elapsed=13.9s
[predict] offset=0 / 50000 elapsed=0.5s
[predict] offset=25600 / 50000 elapsed=13.9s
[predict] offset=0 / 50000 elapsed=0.5s
[predict] offset=25600 / 50000 elapsed=13.9s
[predict] offset=0 / 50000 elapsed=0.5s
[predict] offset=25600 / 50000 elapsed=13.9s
[predict] offset=0 / 50000 elapsed=0.5s
[predict] offset=25600 / 50000 elapsed=13.9s
[predict] offset=0 / 50000 elapsed=0.5s
[predict] offset=25600 / 50000 elapsed=14.3s
[P_B2_raw_center_strong] finished shard 1/8
[P_B2_raw_center_strong] finished shard 2/8
[P_B2_raw_center_strong] step=200/695 obj=0.076828 main=0.068877 center_pen=0.015902
[P_B2_raw_center_strong] finished shard 3/8
[P_B2_raw_center_strong] finished shard 4/8
[P_B2_raw_center_strong] step=400/695 obj=0.077004 main=0.069370 center_pen=0.015269
[P_B2_raw_center_strong] finished shard 5/8
[P_B2_raw_cente

,epoch,train_objective,train_main_term,train_center_penalty,oracle_gate_score,oracle_stable_0.7_slope,oracle_midband_mae_sum_stable,oracle_center_amp_ratio,oracle_center_false_0.1eq,oracle_center_false_0.2eq,pooled_center_amp_ratio,pooled_center_false_0.1eq,pooled_center_false_0.2eq,center_proxy_active_rate,center_proxy_active_weight_mean,lr,epoch_time_sec,variant
0,0,0.076298,0.068939,0.014717,1.306110,0.597325,0.596733,6.296172,0.588235,0.264706,4.354734,0.409091,0.181818,0.000000,0.00000,0.000001,795.404239,P_B2_raw_center_strong
1,0,0.068061,0.067954,0.000214,1.396355,0.629220,0.596832,7.044562,0.647059,0.294118,4.853208,0.454545,0.227273,0.351769,0.84859,0.000001,795.666219,P_B1_proxy_center_weighted


## Final Comparison

Tong hop:

- existing checkpoints
- pilot checkpoints moi

Va xep hang theo `failure_b_score`, khong theo `MSE`.

In [ ]:
pilot_registry = [
    {"label": name, "checkpoint": str(result["best_checkpoint"]), "target_scale": 600.0}
    for name, result in pilot_results.items()
]
combined_registry = EXISTING_REGISTRY + pilot_registry
combined_eval = lab.evaluate_failure_b_registry(
    registry=combined_registry,
    pooled_center_bundle=pooled_center_bundle,
    oracle_bundle=ORACLE_BUNDLE,
    data_root=DATA_ROOT,
    train_cfg=EVAL_CFG,
    oracle_cfg=ORACLE_CFG,
    paths=paths,
    device=DEVICE,
    prefix="combined_failure_b",
)
display(combined_eval["primary"])

[test_600] offset=0 / 200000 elapsed=0.5s
[test_600] offset=25600 / 200000 elapsed=13.8s
[test_600] offset=51200 / 200000 elapsed=27.2s
[test_600] offset=76800 / 200000 elapsed=40.5s
[test_600] offset=102400 / 200000 elapsed=53.8s
[test_600] offset=128000 / 200000 elapsed=67.0s
[test_600] offset=153600 / 200000 elapsed=80.3s
[test_600] offset=179200 / 200000 elapsed=93.6s
[oracle_subset_600] offset=0 / 240 elapsed=0.1s
[test_600] offset=0 / 200000 elapsed=0.5s
[test_600] offset=25600 / 200000 elapsed=13.8s
[test_600] offset=51200 / 200000 elapsed=27.1s
[test_600] offset=76800 / 200000 elapsed=40.4s
[test_600] offset=102400 / 200000 elapsed=53.7s
[test_600] offset=128000 / 200000 elapsed=67.0s
[test_600] offset=153600 / 200000 elapsed=80.3s
[test_600] offset=179200 / 200000 elapsed=93.5s
[oracle_subset_600] offset=0 / 240 elapsed=0.1s
[test_600] offset=0 / 200000 elapsed=0.5s
[test_600] offset=25600 / 200000 elapsed=13.8s
[test_600] offset=51200 / 200000 elapsed=27.1s
[test_600] offset=

,label,target_scale,metric_scale,test_mse_0.1eq,test_mse_0.2eq,test_mse_0.5eq,test_mse_0.7eq,test_slope_0.1eq,test_slope_0.2eq,test_slope_0.7eq,...,oracle_band_sign_0.2_0.5_stable,selection_score_v2,pooled_center_mae,pooled_center_amp_ratio,pooled_center_false_0.1eq,pooled_center_false_0.2eq,pooled_center_wrong_sign_0.1eq,pooled_center_wrong_sign_0.2eq,pooled_center_spread_ratio,failure_b_score
0,A2_band_balanced,600.0,600.0,0.029684,0.028911,0.036876,0.050254,1.283758,0.898517,0.594372,...,1.000000,1.462451,0.092978,4.145144,0.363636,0.136364,0.045455,0.0,4.354346,0.617330
1,L0_control_hybrid,600.0,600.0,0.031418,0.030474,0.038108,0.051023,1.336287,0.927135,0.608344,...,1.000000,1.471977,0.092319,4.189385,0.363636,0.136364,0.045455,0.0,4.406538,0.618233
2,baseline,600.0,600.0,0.032140,0.031037,0.038544,0.051428,1.338430,0.923810,0.605962,...,0.933333,1.502943,0.094196,4.317992,0.409091,0.136364,0.045455,0.0,4.468379,0.650608
3,L1_z_strong_hybrid,600.0,600.0,0.034724,0.033309,0.040471,0.052911,1.366364,0.941777,0.617748,...,1.000000,1.478411,0.094821,4.255428,0.454545,0.136364,0.045455,0.0,4.459377,0.653007
4,L4_A1_plus_A2,600.0,600.0,0.039988,0.038093,0.044461,0.055779,1.436736,0.987802,0.645502,...,1.000000,1.551388,0.108972,4.799236,0.500000,0.181818,0.090909,0.0,5.045348,0.732969
5,S1_A1_center_w020_m010,600.0,600.0,0.041476,0.039600,0.045768,0.056668,1.490324,1.023160,0.661409,...,1.000000,1.577594,0.109851,4.897958,0.500000,0.181818,0.090909,0.0,5.114964,0.742564
6,P_B2_raw_center_strong,600.0,600.0,0.038979,0.037342,0.043920,0.055345,1.446638,0.998386,0.645217,...,1.000000,1.574144,0.109249,4.835386,0.545455,0.227273,0.090909,0.0,4.961691,0.762453
7,L3_full_A1,600.0,600.0,0.047132,0.044530,0.049749,0.059749,1.575222,1.070809,0.683185,...,1.000000,1.633412,0.121076,5.379499,0.500000,0.227273,0.090909,0.0,5.547908,0.804864
8,P_B1_proxy_center_weighted,600.0,600.0,0.047732,0.045042,0.050225,0.060191,1.574411,1.071503,0.683261,...,1.000000,1.675054,0.124860,5.490602,0.590909,0.272727,0.090909,0.0,5.541981,0.855908


: 